# Stage 18A: fold-safe branch retrieval benchmark

Stage 17 strong baseへ、別wellの近接trajectoryから得る地層面Uを小さくblendします。donorは評価wellと異なるfoldだけに限定し、geometry・GR・visible-prefix calibrationを固定subsetでablationします。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir()
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 固定artifact

Stage 16B donor graph、Stage 17A public OOF、Stage 17B always-selectorを使用します。Stage 17C gateは使用しません。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage17a_run = artifact_dir / 'stage17_public_replay_full_v002'
stage17b_run = artifact_dir / 'stage17b_selector_replay_full_v001'
assert (stage16b_run / 'donor_graph.parquet').is_file()
assert (stage17a_run / 'replay_predictions.parquet').is_file()
assert (stage17b_run / 'selector_predictions.parquet').is_file()
print(stage16b_run, stage17a_run, stage17b_run, sep='\n')

## 固定150-cut retrieval benchmark

各standard fold × prefix fractionからSHA-256で6 cutsを固定抽出します。standard/spatial/branch-groupの各familyで同じfold donorを除外します。

In [ ]:
RUN_ID = 'stage18a_branch_retrieval_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-branch-retrieval',
        '--config', 'configs/experiment/stage18a_branch_retrieval.yaml',
        '--stage16b-run', str(stage16b_run), '--stage17a-run', str(stage17a_run),
        '--stage17b-run', str(stage17b_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage18a_complete': summary['stage18a_complete'],
    'promoted_to_all_cut_retrieval': summary['promoted_to_all_cut_retrieval'],
    'sample_cuts': summary['sample_cuts'],
    'primary_profile': summary['primary_profile'],
    'standard_primary': summary['reports']['standard__' + summary['primary_profile']],
    'spatial_primary': summary['reports']['spatial__' + summary['primary_profile']],
    'branch_primary': summary['reports']['branch__' + summary['primary_profile']],
    'ablation_reports': summary['reports'],
    'bootstrap_95pct': summary['bootstrap_95pct'],
    'gates': summary['gates'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。fixed profileは`prefix_gr_w020`で、同じ結果を見て係数を変更しません。